# DH Transformation Matrix — Modified DH Convention

**Source:** CMU 16-311 Lecture 16b  
**Reference:** https://www.cs.cmu.edu/~16311/f05/lecture/lec16b.html  
**Textbook:** Craig, *Introduction to Robotics*, Ch. 3

---

## What is a Transformation Matrix?

Each joint in a robot arm defines a new coordinate frame. A **transformation matrix** describes how to get from one frame to the next — it encodes both a rotation and a translation in a single 4×4 matrix.

Think of it like directions: "from where you are, rotate this way, then walk this far." Chaining all the joints together gives you the position and orientation of the end-effector.

The 4×4 structure is called a **homogeneous transformation matrix**:

$$
T = \begin{bmatrix} R & p \\ 0 & 1 \end{bmatrix}
$$

Where:
- $R$ is a 3×3 rotation matrix
- $p$ is a 3×1 translation vector
- The bottom row $[0, 0, 0, 1]$ is always fixed

---

## The Modified DH Transformation Matrix

In the **modified DH convention**, the transformation from frame $i-1$ to frame $i$ is built from four elementary operations applied in this order:

1. Rotate about $x_{i-1}$ by $\alpha_{i-1}$
2. Translate along $x_{i-1}$ by $a_{i-1}$
3. Rotate about $z_i$ by $\theta_i$
4. Translate along $z_i$ by $d_i$

Combined, this gives:

$$
^{i-1}T_i =
\begin{bmatrix}
\cos\theta_i & -\sin\theta_i & 0 & a_{i-1} \\
\sin\theta_i \cos\alpha_{i-1} & \cos\theta_i \cos\alpha_{i-1} & -\sin\alpha_{i-1} & -d_i \sin\alpha_{i-1} \\
\sin\theta_i \sin\alpha_{i-1} & \cos\theta_i \sin\alpha_{i-1} & \cos\alpha_{i-1} & d_i \cos\alpha_{i-1} \\
0 & 0 & 0 & 1
\end{bmatrix}
$$

---

## Parameter Reminder

| Parameter | Role in matrix | Fixed or variable? |
|---|---|---|
| $a_{i-1}$ | Translation along $x$ | Fixed (link geometry) |
| $\alpha_{i-1}$ | Rotation about $x$ | Fixed (link geometry) |
| $d_i$ | Translation along $z$ | **Variable** if prismatic |
| $\theta_i$ | Rotation about $z$ | **Variable** if revolute |

---

## Forward Kinematics — Chaining T Matrices

To find the position and orientation of the end-effector, multiply all joint matrices together:

$$
^0T_n = {}^0T_1 \cdot {}^1T_2 \cdot {}^2T_3 \cdots {}^{n-1}T_n
$$

The result is a single 4×4 matrix where:
- The top-left 3×3 block is the **orientation** of the end-effector
- The top-right 3×1 column is the **position** of the end-effector

> **Order matters:** matrix multiplication is not commutative. Always multiply left to right, joint 1 through joint n.

---

## What to implement in `transforms.py`

1. A function that takes one joint's DH parameters and returns its 4×4 T matrix
2. A function that takes all joints and multiplies the matrices together (forward kinematics)


## Example — Building One T Matrix with NumPy

In [ ]:
import numpy as np

def dh_matrix(a, alpha, d, theta):
    """Returns the 4x4 modified DH transformation matrix for one joint."""
    return np.array([
        [np.cos(theta), -np.sin(theta), 0, a],
        [np.sin(theta)*np.cos(alpha), np.cos(theta)*np.cos(alpha), -np.sin(alpha), -d*np.sin(alpha)],
        [np.sin(theta)*np.sin(alpha), np.cos(theta)*np.sin(alpha),  np.cos(alpha),  d*np.cos(alpha)],
        [0, 0, 0, 1]
    ])

# Example: joint 1 from example_6dof.yaml
T1 = dh_matrix(a=0, alpha=0, d=1, theta=np.pi/2)
print(T1)

## Example — Chaining T Matrices (Forward Kinematics)

In [ ]:
# Suppose you have a list of T matrices, one per joint
# np.linalg.multi_dot multiplies them all together in order

T_matrices = [T1]  # add more joints here

T_total = np.linalg.multi_dot(T_matrices)
print("End-effector position:", T_total[:3, 3])
print("End-effector orientation:\n", T_total[:3, :3])